# Books to Scrape - Webscraping 

### Gameplan:

Scraping data from [Books to Scrape](https://books.toscrape.com/) via `requests` and `BeatifulSoup` to find messy data. 

Then this data will be cleaned, organized and ordered to create a before and after Visualization set in another Notekbook


In [2]:
# Imports
# Data Handling and Storage
import pandas as pd
import numpy as np

# scraping tools
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

### Webscraping 

In [15]:
# storing the page to scrape
url = "https://books.toscrape.com/index.html"

# setting up try-except function to prevent errors
print('Fetching Website...')
try:
    page = requests.get(url)
    page.raise_for_status()  # -> error if page is not found
    print(f'Success! Status Code: { page.status_code}')
except requests.exceptions.RequestException as e:
    print(f'Error fetching page: {e}')
    exit()

Fetching Website...
Success! Status Code: 200


In [16]:
# setting up the kitchen -> dumb AI joke haha
# creating BeatifulSoup object to parse HTML
soup = BeautifulSoup(page.content, 'html.parser')

# checking it it worked 
soup.title.text

'\n    All products | Books to Scrape - Sandbox\n'

In [ ]:
# creating the empty list
data = []

# creating a dictionary to change the string star rating into numbers
rating_to_nums = {
    "One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5
    }

# using a while loop to get all books on the website
while url: 
    print(f'Scraping {url}')  # show progress

    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # finding all books 
    books = soup.find_all('article', class_='product_pod')

    # looping through every book found on the page
    for book in books:

        # extracting raw data
        title = book.find('h3').find('a')['title']
        rating_classes = book.find('p', class_='star-rating')['class']
        price_raw = book.find('p', class_='price_color').text
        stock_tag = book.find('p', class_='instock availability')

        # first cleaning
        rating_clean = rating_to_nums[rating_classes[1]]
        # using more complex logic to clean the price column
        price_clean = float("".join(c for c in price_raw if c.isdigit() or c == '.'))
        is_available = (stock_tag.text.strip() == "In stock")

        # storing the books
        data.append({
            'title': title,
            'rating': rating_clean,
            'price': price_clean,
            'available': is_available
            })

    # Moving to next page
    next_page = soup.find('li', class_='next')

    # extracting href if it exists
    if next_page:
        next_url = next_page.find('a')['href']
        url = urljoin(url, next_url)  # merging urls
    else:
        print('No next page. Scraping Finished.')
        url = None

# storing the output in a dataframe
book_data = pd.DataFrame(data)

print(f'Total books scraped: {len(book_data)}')

In [ ]:
# displazing the first entries
book_data.head()

,title,rating,price,available
0,A Light in the Attic,3,51.77,True
1,Tipping the Velvet,1,53.74,True
2,Soumission,1,50.10,True
3,Sharp Objects,4,47.82,True
4,Sapiens: A Brief History of Humankind,5,54.23,True


In [18]:
print(book_data.describe())

            rating       price
count  1000.000000  1000.00000
mean      2.923000    35.07035
std       1.434967    14.44669
min       1.000000    10.00000
25%       2.000000    22.10750
50%       3.000000    35.98000
75%       4.000000    47.45750
max       5.000000    59.99000


In [17]:
# Storing the DataFrame for future me
book_data.to_csv("books.toscrape_20251214.csv", index=False)